In [35]:
import os
import sys
import pyarrow as pa
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from joblib import Parallel, delayed
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
sys.path.insert(0, "..")
from paths import resolve

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
_NCPU = os.cpu_count() or 1
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | pyarrow {pa.__version__}", flush=True)

Running with 8 CPU cores | pyarrow 24.0.0


Variables

In [ ]:
%run 0_variables.ipynb

Remove duplicate features (post-ranking)

In [42]:
_stem = Path(os.environ['FEATURE_DATASET']).stem
feature_data = pd.read_parquet(
    resolve(f"4_Features select/Selected_features/{os.environ['TARGET']}_feature_data_{_stem}.parquet")
)

features = pd.read_parquet(os.environ["FEATURES_PROCESSED_PATH"])

def remove_duplicate_features(feature_data, features_full):
    LINEAR_CORR_THRESHOLD = 0.95     # |Pearson| above this → linear duplicate
    NONLINEAR_CORR_THRESHOLD = 0.95  # |Spearman| above this → monotonic non-linear duplicate

    ranked_features = feature_data["feature"].tolist()  # already sorted best → worst by mean_mi

    cols_present = [f for f in ranked_features if f in features_full.columns]
    features_full = features_full[cols_present]

    n_rows_total = len(features_full)
    n_rows = min(int(os.environ["DEDUP_SUBSAMPLE_AMOUNT"]), n_rows_total)
    if n_rows < n_rows_total:
        rng = np.random.default_rng(42)
        subsample_idx = np.sort(rng.choice(n_rows_total, size=n_rows, replace=False))
        features_full = features_full.iloc[subsample_idx]

    n_rows, n_features = features_full.shape
    print(
        f"Standardising {n_rows:,} rows × {n_features} features "
        f"(subsample of {n_rows_total:,} total rows)...",
        flush=True,
    )

    # Standardise columns → Pearson corr = (Z.T @ Z) / n_rows per column pair
    col_means = features_full.mean()
    col_stds = features_full.std().replace(0, 1)
    Z_pearson = ((features_full - col_means) / col_stds).values.astype(np.float32)

    # Rank then standardise → Spearman via same dot product trick
    ranks = features_full.rank()
    rank_means = ranks.mean()
    rank_stds = ranks.std().replace(0, 1)
    Z_spearman = ((ranks - rank_means) / rank_stds).values.astype(np.float32)

    col_index = {col: i for i, col in enumerate(cols_present)}

    kept_indices = []  # column positions of kept features in Z_pearson / Z_spearman
    duplicate_flags = {}

    for feature in tqdm(ranked_features, desc="Deduplicating features", leave=True):
        if feature not in col_index:
            # feature absent from the loaded parquet — keep unconditionally
            duplicate_flags[feature] = False
            continue
        fi = col_index[feature]
        if not kept_indices:
            duplicate_flags[feature] = False
            kept_indices.append(fi)
            continue

        kept_arr = np.array(kept_indices)
        # Dot candidate column against each kept column → correlation vector; O(n_kept × n_rows)
        pearson_vals = np.abs(Z_pearson[:, fi] @ Z_pearson[:, kept_arr]) / n_rows
        spearman_vals = np.abs(Z_spearman[:, fi] @ Z_spearman[:, kept_arr]) / n_rows
        is_dup = bool(pearson_vals.max() > LINEAR_CORR_THRESHOLD or spearman_vals.max() > NONLINEAR_CORR_THRESHOLD)
        duplicate_flags[feature] = is_dup
        if not is_dup:
            kept_indices.append(fi)

    feature_data_flagged = feature_data.copy()
    feature_data_flagged["is_duplicate"] = feature_data_flagged["feature"].map(duplicate_flags)

    n_kept = feature_data_flagged["is_duplicate"].eq(False).sum()
    n_removed = feature_data_flagged["is_duplicate"].eq(True).sum()
    print(
        f"Kept {n_kept} unique features, flagged {n_removed} duplicates "
        f"(Pearson threshold={LINEAR_CORR_THRESHOLD}, Spearman threshold={NONLINEAR_CORR_THRESHOLD})",
        flush=True,
    )

    feature_data_unique = feature_data_flagged[feature_data_flagged["is_duplicate"] == False].reset_index(drop=True)

    display(feature_data_unique[:3])

    return feature_data_unique, feature_data_flagged, features_full


feature_data_unique, feature_data_flagged, features_subsampled = remove_duplicate_features(feature_data, features)

_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
feature_data_flagged.to_parquet(
    resolve(f"4_Features select/Selected_features/{_target}_feature_data_flagged_{_stem}.parquet")
)
feature_data_unique.to_parquet(
    resolve(f"4_Features select/Selected_features/{_target}_feature_data_unique_{_stem}.parquet")
)
features_subsampled.to_parquet(
    resolve(f"4_Features select/Selected_features/{_target}_features_subsampled_{_stem}.parquet")
)


In [43]:
_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
feature_data_flagged = pd.read_parquet(
    resolve(f"4_Features select/Selected_features/{_target}_feature_data_flagged_{_stem}.parquet")
)

_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
features_subsampled = pd.read_parquet(
    resolve(f"4_Features select/Selected_features/{_target}_features_subsampled_{_stem}.parquet")
)

import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

TOP_N = 20

# Top N duplicate features by mean MI (highest MI duplicates — most "interesting" redundancies)
duplicate_features = (
    feature_data_flagged[feature_data_flagged["is_duplicate"] == True]
    .head(TOP_N)["feature"]
    .tolist()
)
top_features = [f for f in duplicate_features if f in features_subsampled.columns]
corr = features_subsampled[top_features].corr(method="pearson")

# Diverging colormap with accent at 75th percentile (0.5 on the -1→1 scale = position 0.75)
# blue → navy (mid) → electric cyan accent → crimson
_cmap = LinearSegmentedColormap.from_list(
    "dark_div_accent",
    [
        (0.00, "#1565C0"),   # -1.0  deep blue
        (0.50, "#0D1B2A"),   #  0.0  dark navy (midpoint)
        (0.75, "#FFB300"),   # +0.5  electric cyan accent (75th percentile)
        (1.00, "#C62828"),   # +1.0  deep crimson
    ],
)

with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(12, 10), facecolor="#1e1e1e")
    ax.set_facecolor("#1e1e1e")
    im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap=_cmap, aspect="auto")
    cbar = plt.colorbar(im, ax=ax, label="Pearson correlation")
    cbar.ax.yaxis.label.set_color("white")
    cbar.ax.tick_params(colors="white")
    ax.set_xticks(range(len(top_features)))
    ax.set_yticks(range(len(top_features)))
    ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8, color="white")
    ax.set_yticklabels(corr.index, fontsize=8, color="white")
    ax.set_title(f"Pearson correlation — top {len(top_features)} duplicate features (by mean MI)", fontsize=11, color="white")
    ax.tick_params(colors="white")
    for spine in ax.spines.values():
        spine.set_edgecolor("#444444")
    plt.tight_layout()
    plt.show()


In [44]:
_stem = Path(os.environ['FEATURE_DATASET']).stem
_target = os.environ['TARGET']
feature_data_flagged = pd.read_parquet(
    resolve(f"4_Features select/Selected_features/{_target}_feature_data_flagged_{_stem}.parquet")
)

import matplotlib.pyplot as plt

df_plot = feature_data_flagged.sort_values("mean_mi", ascending=False).reset_index(drop=True)
df_plot["rank_pos"] = range(1, len(df_plot) + 1)

unique_mi = df_plot["mean_mi"].where(df_plot["is_duplicate"] == False, 0)
dup_mi = df_plot["mean_mi"].where(df_plot["is_duplicate"] == True, 0)

with plt.style.context("dark_background"):
    fig, ax = plt.subplots(figsize=(14, 5), facecolor="#1e1e1e")
    ax.set_facecolor("#1e1e1e")

    ax.fill_between(df_plot["rank_pos"], unique_mi, alpha=0.85, color="#1565C0", label="Unique")
    ax.fill_between(df_plot["rank_pos"], dup_mi, alpha=0.75, color="#C62828", label="Duplicate (removed)")

    ax.set_xlabel("Feature rank (most unique → least)", color="white", fontsize=10)
    ax.set_ylabel("Mean mutual information", color="white", fontsize=10)
    ax.set_title("Feature uniqueness by MI rank", fontsize=11, color="white")
    ax.tick_params(colors="white")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["bottom"].set_edgecolor("#444444")
    ax.spines["left"].set_edgecolor("#444444")
    ax.legend(facecolor="#2a2a2a", edgecolor="#444444", labelcolor="white")
    plt.tight_layout()
    plt.show()
